# 00 — Patent Search & Filter
Side quest: explore and filter the Excel dataset before running the pipeline.
Use this notebook to find the 90 patents you want to work with.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import yaml
from src.patents import load_patents

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

df = load_patents(cfg)
print(f"Total patents: {len(df)}")
df.head(3)

## Explore columns

In [ ]:
# What industries / domains are in the dataset?
print("Record Types:")
print(df["Record Type"].value_counts())
print()
print("Legal Status:")
print(df["Family Legal Status(Dead/Alive)"].value_counts())
print()
print("Tech Sub Domain (top 10):")
print(df["Tech Sub Domain"].value_counts().head(10))

## Filter to your target subset
The filter is driven by `config.yaml → subset`. To change the target CPC codes (or any other filter), edit `config.yaml` and re-run this cell. No code changes needed.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.patents import load_patents, get_subset
from src.config_loader import load_config

cfg = load_config()
df, missing_df = load_patents(cfg)
print(f"Total patents loaded: {len(df)}")

# Subset is now fully driven by config.yaml → subset section
filtered = get_subset(df, cfg)
print(f"Filtered: {len(filtered)} patents")
filtered[["Record Number", "Title", "CPC", "Record Type", "Family Legal Status(Dead/Alive)"]].head(10)

## Save your filtered list
The filtered subset is written to `extraction_status.xlsx → sheet "00"` in your logs folder — the same file notebook 01 writes to. Re-run this cell after changing filters in `config.yaml`.

In [ ]:
from pathlib import Path

status_path = Path(cfg["paths"]["logs"]) / "extraction_status.xlsx"
status_path.parent.mkdir(parents=True, exist_ok=True)

cols = ["Record Number", "Title", "CPC", "Record Type",
        "Family Legal Status(Dead/Alive)", "Tech Sub Domain"]
save_cols = [c for c in cols if c in filtered.columns]

write_mode = "a" if status_path.exists() else "w"
writer_kwargs = {"mode": write_mode, "engine": "openpyxl"}
if write_mode == "a":
    writer_kwargs["if_sheet_exists"] = "replace"

with pd.ExcelWriter(status_path, **writer_kwargs) as writer:
    filtered[save_cols].to_excel(writer, sheet_name="00", index=False)

print(f"Saved {len(filtered)} patents → sheet '00' in:\n  {status_path}")